# 02 – Retrieve, Rerank & Generate (Grounded RAG)

This notebook demonstrates a full **Retrieval-Augmented Generation (RAG)** pipeline using:
- **FAISS** – vector similarity search for initial retrieval
- **Cohere Rerank** – cross-encoder reranking to improve relevance
- **Cohere Command** – grounded generation with cited sources

Interim results are printed at each stage so you can inspect how the pipeline works.

> **Prerequisites**  
> - Run `01_embed_documents.ipynb` first to create the `./faiss_index/` folder.  
> - Set `COHERE_API_KEY` in your environment or `.env` file.

## 1. Imports and configuration

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_cohere import ChatCohere, CohereEmbeddings, CohereRerank
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

from config import FAISS_INDEX, EMBED_MODEL, RERANK_MODEL, GENERATION_MODEL, RETRIEVAL_K, RERANK_TOP_N
from helper import get_cohere_api_key, check_faiss_store_existence

# Load environment variables from .env if present
load_dotenv()

cohere_api_key = get_cohere_api_key()
check_faiss_store_existence()

print("✅ Configuration OK")

## 2. Load the FAISS vector store

In [ ]:
embeddings = CohereEmbeddings(
    model=EMBED_MODEL,
    cohere_api_key=cohere_api_key,
)

vector_store = FAISS.load_local(
    str(FAISS_INDEX),
    embeddings,
    allow_dangerous_deserialization=True,
)

print(f"📦 FAISS index loaded from '{FAISS_INDEX}'")

## 3. Define the query

In [ ]:
# ✏️  Change this to any question you want to ask about your documents
QUERY = "How to build a toilet?"

print(f"🤔 Query: {QUERY}")

## 4. Step 1 – Vector similarity retrieval

We perform a **semantic similarity search** against the FAISS index to get the top-K candidate chunks.

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": RETRIEVAL_K})

retrieved_docs = retriever.invoke(QUERY)

# ---------------------------------------------------------------------

print(f"🔍 Retrieved {len(retrieved_docs)} candidate(s) from FAISS\n")
print("=" * 70)
for i, doc in enumerate(retrieved_docs, 1):
    source = doc.metadata.get("source", "unknown")
    page   = doc.metadata.get("page", "?")
    print(f"\n[{i}] Source: {source} | Page: {page}")
    print("-" * 70)
    print(doc.page_content[:400])
print("\n" + "=" * 70)

## 5. Step 2 – Cohere Rerank

We feed all retrieved candidates through **Cohere Rerank** (a cross-encoder model) to re-score them by actual relevance to the query and keep only the top-N.

In [ ]:
reranker = CohereRerank(
    model=RERANK_MODEL,
    cohere_api_key=cohere_api_key,
    top_n=RERANK_TOP_N,
)

reranked_docs = reranker.compress_documents(retrieved_docs, QUERY)

# ---------------------------------------------------------------------

print(f"🏆 After reranking – top {len(reranked_docs)} document(s)\n")
print("=" * 70)
for i, doc in enumerate(reranked_docs, 1):
    source        = doc.metadata.get("source", "unknown")
    page          = doc.metadata.get("page", "?")
    relevance     = doc.metadata.get("relevance_score", "n/a")
    print(f"\n[{i}] Source: {source} | Page: {page} | Relevance score: {relevance:.4f}" if isinstance(relevance, float) else f"\n[{i}] Source: {source} | Page: {page} | Relevance score: {relevance}")
    print("-" * 70)
    print(doc.page_content[:400])
print("\n" + "=" * 70)

## 6. Step 3 – Grounded generation with Cohere Command

We build a prompt that includes the reranked context chunks and asks **Cohere Command** to answer the query while grounding its response in the provided documents.

In [ ]:
RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant that answers questions strictly based on the "
     "provided context documents. "
     "If the answer cannot be found in the documents, say so clearly. "
     "Always cite which document(s) support your answer."),
    ("human",
     "Context documents:\n\n{context}\n\n"
     "Question: {question}"),
])

llm = ChatCohere(
    model=GENERATION_MODEL,
    cohere_api_key=cohere_api_key,
    temperature=0.3,
)

# ---------------------------------------------------------------------

print("🤖 LLM configured:", GENERATION_MODEL)

In [ ]:
from helper import format_docs

# Show the context that will be injected into the prompt
context_str = format_docs(reranked_docs)

# ---------------------------------------------------------------------

print("📋 Context passed to the LLM:")
print("=" * 70)
print(context_str)
print("=" * 70)

In [ ]:
# Build and invoke the RAG chain
rag_chain = (
    {"context": RunnablePassthrough(), "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke({"context": context_str, "question": QUERY})

# ---------------------------------------------------------------------

print("💬 Answer:")
print("=" * 70)
print(answer)
print("=" * 70)